In [1]:
!pip install git+https://github.com/LeoLe12/Federated-Risk-Aware-Projection.git
!pip install flwr
!pip install -U "flwr[simulation]"

  Cloning https://github.com/LeoLe12/Federated-Risk-Aware-Projection.git to /tmp/pip-req-build-9z56h5nl
  Running command git clone --filter=blob:none --quiet https://github.com/LeoLe12/Federated-Risk-Aware-Projection.git /tmp/pip-req-build-9z56h5nl
  Resolved https://github.com/LeoLe12/Federated-Risk-Aware-Projection.git to commit 5c5496af6e20516d6a4b6064d087e4c199d07332
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
INFO: pip is looking at multiple versions of typer-slim to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 90.6 MB/s eta 0:00:00:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 32.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 926.2/926.2 kB 50.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 98.1 MB/s eta 0:00:00:00:01
 

In [2]:
from rap_fl.client import RAPTrainer
from rap_fl.server import RAPStrategy

print("✅ Wrapper importati con successo! La libreria RAP-FL è operativa.")

✅ Wrapper importati con successo! La libreria RAP-FL è operativa.


In [5]:
from huggingface_hub import login

from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("HFkey")


# Effettua il login automatico usando il token
login(token=secret_value_0)

# MODEL

In [6]:
import os
import gc
import torch
import flwr as fl
import pandas as pd
from transformers import AutoTokenizer, AutoConfig, AutoModelForCausalLM, BitsAndBytesConfig, TrainingArguments, DataCollatorForLanguageModeling
from datasets import Dataset
from peft import LoraConfig, get_peft_model
from rap_fl.client import RAPTrainer   
from rap_fl.server import RAPStrategy  

# 1. SETUP TOKENIZER
model_id = "EleutherAI/deep-ignorance-pretraining-stage-unfiltered"
tokenizer = AutoTokenizer.from_pretrained(model_id)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token



# DATASET

In [7]:
# 2. SETUP DATASET (Sostituisci con i tuoi percorsi reali se diversi)
# Percorso della cartella contenente i dataset
base_path = '/kaggle/input/datasets/leolei12/training-datasets/TRAINING_DATASET' 

# Caricamento dei dataset
df_safe = pd.read_csv(f'{base_path}/safe_dataset.csv').sample(n=100, random_state=42)
df_threat = pd.read_csv(f'{base_path}/threat_dataset.csv').sample(n=100, random_state=42)

def tokenize_function(example):
    prompt = f"Question:\nTopic: {example['topic;']}\nSub-topic: {example['sub_topic']}\n\n{example['message_1']}\n\nAnswer: "
    tokenized = tokenizer(prompt + example['message_2'] + tokenizer.eos_token, truncation=True, padding="max_length", max_length=128)
    tokenized['labels'] = tokenized['input_ids'].copy()
    return tokenized

dataset_safe = Dataset.from_pandas(df_safe).map(tokenize_function).rename_column("prob_class_1", "risk_score")
dataset_threat = Dataset.from_pandas(df_threat).map(tokenize_function).rename_column("prob_class_1", "risk_score")

client_datasets = {"0": dataset_safe, "1": dataset_threat}

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

# CLIENT

In [8]:
import gc

class RAPFlowerClient(fl.client.NumPyClient):
    def __init__(self, cid, model, train_dataset):
        self.cid = cid
        self.model = model
        self.train_dataset = train_dataset

    def set_parameters(self, parameters):
        trainable_keys = [k for k, v in self.model.named_parameters() if v.requires_grad]
        state_dict = self.model.state_dict()
        for k, v in zip(trainable_keys, parameters):
            state_dict[k] = torch.tensor(v).to(self.model.device).to(torch.float16)
        self.model.load_state_dict(state_dict, strict=False)

    def fit(self, parameters, config):
        print(f"\n--- Avvio Training su Client {self.cid} ---")
        self.set_parameters(parameters)

        # Trucchi salva-VRAM per l'addestramento
        from transformers import TrainingArguments
        training_args = TrainingArguments(
            output_dir=f"./results_client_{self.cid}",
            per_device_train_batch_size=1,
            gradient_accumulation_steps=4,
            gradient_checkpointing=True,
            optim="paged_adamw_8bit",
            max_steps=1, 
            logging_steps=1,
            report_to="none"
        )
        
        trainer = RAPTrainer(
            model=self.model,
            args=training_args,
            train_dataset=self.train_dataset,
            data_collator=DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)
        )
        
        # Addestramento
        trainer.train()
        
        # Estrazione dei gradienti/pesi modificati
        delta_U_ndarrays = [val.numpy() for val in trainer.delta_U.values()]
        delta_R_ndarrays = [val.numpy() for val in trainer.delta_R.values()]
        
        # ==========================================
        # LA TUA IDEA: PULIZIA POST-TRAINING
        # ==========================================
        print(f"🧹 Pulizia VRAM per il Client {self.cid} in corso...")
        # 1. Distruggiamo i riferimenti al modello e al trainer
        del self.model
        del trainer
        
        # 2. Forziamo il Garbage Collector di Python
        gc.collect()

        torch.cuda.empty_cache()
        torch.cuda.synchronize()
        
        # 3. Svuotiamo fisicamente la memoria CUDA
        torch.cuda.empty_cache()
        print(f"✅ VRAM liberata! GPU pronta per il prossimo task.")
        # ==========================================
        
        return delta_U_ndarrays + delta_R_ndarrays, len(self.train_dataset), {}

def client_fn(cid: str):
    # Approccio Stateless: Carichiamo il modello ex-novo ogni volta
    print(f"⏳ Client {cid}: Caricamento del modello a 4-bit...")
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
    )
    local_base = AutoModelForCausalLM.from_pretrained(
        model_id,
        quantization_config=bnb_config, 
        device_map={"": 0} 
    )
    local_base.config.pad_token_id = local_base.config.eos_token_id
    
    # Abilitiamo il gradient checkpointing per risparmiare RAM
    if hasattr(local_base, "gradient_checkpointing_enable"):
        local_base.gradient_checkpointing_enable()
        
    local_peft = get_peft_model(local_base, lora_cfg)
    for name, param in local_peft.named_parameters():
        if "lora" in name: param.requires_grad = True
            
    return RAPFlowerClient(cid=cid, model=local_peft, train_dataset=client_datasets[cid])

In [9]:
# 3. LOGICA DI ESTRAZIONE DELLE CHIAVI STATICHE
# ==========================================
# Prima di lanciare la simulazione, la strategia ha bisogno dei nomi dei layer.
# Creiamo al volo una lista statica leggendola da una configurazione LoRA fittizia temporanea su CPU
print("📋 Generazione delle chiavi dei parametri per il Server...")
temp_base_model = AutoModelForCausalLM.from_pretrained(model_id, low_cpu_mem_usage=True)
temp_peft_model = get_peft_model(temp_base_model, LoraConfig(r=16, lora_alpha=32, target_modules="all-linear", task_type="CAUSAL_LM"))
trainable_keys = [name for name, param in temp_peft_model.named_parameters() if param.requires_grad]

# Puliamo la memoria CPU subito dopo
del temp_base_model
del temp_peft_model
import gc
gc.collect()

📋 Generazione delle chiavi dei parametri per il Server...


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/388 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

0

# SERVER

In [11]:
import os
import gc
import torch
import flwr as fl
from transformers import AutoConfig, AutoModelForCausalLM, BitsAndBytesConfig, TrainingArguments, DataCollatorForLanguageModeling
from peft import LoraConfig, get_peft_model

# 1. Configura le 2 GPU di Kaggle
os.environ["CUDA_VISIBLE_DEVICES"] = "0,1"

# 2. Estrazione chiavi a costo ZERO con Meta Device
print("📋 Estrazione architettura (Costo VRAM: 0 GB)...")
model_id = "EleutherAI/deep-ignorance-pretraining-stage-unfiltered"
config = AutoConfig.from_pretrained(model_id)

with torch.device("meta"):
    meta_model = AutoModelForCausalLM.from_config(config)

lora_cfg = LoraConfig(r=16, lora_alpha=32, target_modules="all-linear", task_type="CAUSAL_LM")
meta_peft_model = get_peft_model(meta_model, lora_cfg)

trainable_keys = [name for name, param in meta_peft_model.named_parameters() if param.requires_grad]
print(f"✅ Trovate {len(trainable_keys)} chiavi LoRA. Le GPU sono pronte e libere!")

# 3. Setup Strategia del Server
strategy = RAPStrategy(
    keys=trainable_keys,
    lambda_t=0.3, 
    mu_t=0.8, 
    gamma_t=0.1,

    fraction_evaluate=0.0,  # Non interrogare nessun client per la valutazione
    min_evaluate_clients=0
)

📋 Estrazione architettura (Costo VRAM: 0 GB)...
✅ Trovate 256 chiavi LoRA. Le GPU sono pronte e libere!


In [12]:
print("\n🚀 Avvio Simulazione Federata RAP-FL su Dual GPU...")
fl.simulation.start_simulation(
    client_fn=client_fn,
    num_clients=2,
    config=fl.server.ServerConfig(num_rounds=3),
    strategy=strategy,
    # 1.0 assicura che ogni client vada su una GPU separata senza accavallarsi
    client_resources={"num_cpus": 2, "num_gpus": 1.0} 
)
print("🎉 Addestramento Completato con Successo!")


🚀 Avvio Simulazione Federata RAP-FL su Dual GPU...


2026-06-18 18:58:24.712011: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1781809104.932963      58 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1781809104.997185      58 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1781809105.534424      58 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1781809105.534450      58 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1781809105.534453      58 computation_placer.cc:177] computation placer alr

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

...

100%|██████████| 1/1 [00:09<00:00,  9.34s/it]


(ClientAppActor pid=384) 🧹 Pulizia VRAM per il Client 0 in corso...
(ClientAppActor pid=384) ✅ VRAM liberata! GPU pronta per il prossimo task.


INFO :      aggregate_fit: received 2 results and 0 failures
INFO :      configure_evaluate: no clients selected, skipping evaluation
INFO :      
INFO :      [SUMMARY]
INFO :      Run finished 3 round(s) in 72.95s
INFO :      


🎉 Addestramento Completato con Successo!


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


# SALVATAGGIO

In [13]:
print('ciao')

ciao
